# 09 Expected Loss Analysis

Milestone 7 creates Expected Loss portfolio decomposition outputs using the combined PD/LGD/EAD dataset from Milestone 6. This notebook generates datasets and checks only; the final analytical report will be created later.

## 1. Load and Validate Combined PD/LGD/EAD Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except ModuleNotFoundError:
    SEABORN_AVAILABLE = False
    print("seaborn is not installed. Heatmaps and plots will use matplotlib fallbacks.")

In [ ]:
current_dir = Path.cwd()
if (current_dir / "data" / "processed").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "processed").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Could not locate the project data/processed directory from the current working directory.")

processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "reports" / "figures"

processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

input_path = processed_dir / "loans_with_pd_lgd_ead_v1.csv"
loans = pd.read_csv(input_path, low_memory=False)

required_fields = [
    "predicted_pd",
    "lgd_estimate",
    "ead_proxy",
    "expected_loss_prelim",
    "default_flag",
]
useful_segment_fields = ["grade", "sub_grade", "term", "purpose", "issue_d", "issue_year", "loan_status"]

missing_required = [field for field in required_fields if field not in loans.columns]
if missing_required:
    raise ValueError(f"Missing required fields: {missing_required}")

print(f"Loaded: {input_path}")
print(f"Shape: {loans.shape}")
print("Available useful segmentation fields:")
print([field for field in useful_segment_fields if field in loans.columns])

In [ ]:
validation_checks = pd.DataFrame({
    "check": [
        "No missing required fields",
        "predicted_pd between 0 and 1",
        "lgd_estimate between 0 and 1",
        "ead_proxy >= 0",
        "expected_loss_prelim >= 0",
        "expected_loss_prelim <= ead_proxy",
    ],
    "passed": [
        not loans[required_fields].isna().any().any(),
        loans["predicted_pd"].between(0, 1, inclusive="both").all(),
        loans["lgd_estimate"].between(0, 1, inclusive="both").all(),
        loans["ead_proxy"].ge(0).all(),
        loans["expected_loss_prelim"].ge(0).all(),
        (loans["expected_loss_prelim"] <= loans["ead_proxy"] + 1e-8).all(),
    ],
})

display(validation_checks)

if not validation_checks["passed"].all():
    raise ValueError("One or more required validation checks failed. Review validation_checks above.")

## 2. Create Issue Year If Needed

In [ ]:
if "issue_year" not in loans.columns and "issue_d" in loans.columns:
    issue_date = pd.to_datetime(loans["issue_d"], format="%b-%Y", errors="coerce")
    if issue_date.isna().mean() > 0.50:
        issue_date = pd.to_datetime(loans["issue_d"], errors="coerce")
    loans["issue_year"] = issue_date.dt.year
    print("Created issue_year from issue_d.")
elif "issue_year" in loans.columns:
    print("issue_year already exists.")
else:
    print("issue_d and issue_year are unavailable. Issue-year output will be skipped.")

## 3. Portfolio-Level Summary

In [ ]:
total_ead = loans["ead_proxy"].sum()
total_expected_loss = loans["expected_loss_prelim"].sum()

portfolio_summary = pd.DataFrame({
    "metric": [
        "loan_count",
        "total_ead",
        "mean_ead",
        "median_ead",
        "total_expected_loss",
        "mean_expected_loss",
        "median_expected_loss",
        "expected_loss_rate",
        "mean_predicted_pd",
        "median_predicted_pd",
        "mean_lgd_estimate",
        "realized_default_rate",
    ],
    "value": [
        len(loans),
        total_ead,
        loans["ead_proxy"].mean(),
        loans["ead_proxy"].median(),
        total_expected_loss,
        loans["expected_loss_prelim"].mean(),
        loans["expected_loss_prelim"].median(),
        total_expected_loss / total_ead,
        loans["predicted_pd"].mean(),
        loans["predicted_pd"].median(),
        loans["lgd_estimate"].mean(),
        loans["default_flag"].mean(),
    ],
})

display(portfolio_summary)

## 4. Reusable Segment Summary Function

In [ ]:
saved_outputs = {}


def segment_expected_loss_summary(data, segment_column):
    """Create an Expected Loss summary by a segment column."""
    summary = (
        data.groupby(segment_column, dropna=False)
        .agg(
            loan_count=("expected_loss_prelim", "size"),
            default_count=("default_flag", "sum"),
            realized_default_rate=("default_flag", "mean"),
            total_ead=("ead_proxy", "sum"),
            mean_ead=("ead_proxy", "mean"),
            total_expected_loss=("expected_loss_prelim", "sum"),
            mean_expected_loss=("expected_loss_prelim", "mean"),
            mean_predicted_pd=("predicted_pd", "mean"),
            median_predicted_pd=("predicted_pd", "median"),
            mean_lgd_estimate=("lgd_estimate", "mean"),
            median_lgd_estimate=("lgd_estimate", "median"),
        )
        .reset_index()
    )
    summary["expected_loss_rate"] = summary["total_expected_loss"] / summary["total_ead"]

    return summary


def sort_segment_summary(summary, segment_column):
    """Sort segment summaries in a readable way."""
    if segment_column == "grade":
        order = list("ABCDEFG")
        summary[segment_column] = pd.Categorical(summary[segment_column], categories=order, ordered=True)
        return summary.sort_values(segment_column)
    if segment_column == "sub_grade":
        grades = list("ABCDEFG")
        order = [f"{grade}{num}" for grade in grades for num in range(1, 6)]
        summary[segment_column] = pd.Categorical(summary[segment_column], categories=order, ordered=True)
        return summary.sort_values(segment_column)
    if segment_column in ["term", "issue_year", "pd_decile"]:
        return summary.sort_values(segment_column)
    if segment_column == "purpose":
        return summary.sort_values("total_expected_loss", ascending=False)
    return summary.sort_values("total_expected_loss", ascending=False)


def save_segment_output(summary, filename):
    output_path = processed_dir / filename
    summary.to_csv(output_path, index=False)
    saved_outputs[filename] = {"path": output_path, "shape": summary.shape}
    print(f"Saved {filename}: {summary.shape}")
    return output_path


def simple_bar(data, x_col, y_col, title, ylabel, output_name, top_n=None):
    plot_data = data.copy()
    if top_n is not None:
        plot_data = plot_data.head(top_n)
    plt.figure(figsize=(10, 5))
    if SEABORN_AVAILABLE:
        sns.barplot(data=plot_data, x=x_col, y=y_col, color="steelblue")
    else:
        plt.bar(plot_data[x_col].astype(str), plot_data[y_col])
    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / output_name, dpi=150)
    plt.show()

## 5. Expected Loss by Grade

In [ ]:
if "grade" in loans.columns:
    el_by_grade = sort_segment_summary(segment_expected_loss_summary(loans, "grade"), "grade")
    save_segment_output(el_by_grade, "expected_loss_by_grade_v1.csv")
    display(el_by_grade)
    simple_bar(el_by_grade, "grade", "total_expected_loss", "Total Expected Loss by Grade", "Total Expected Loss", "expected_loss_total_by_grade_v1.png")
    simple_bar(el_by_grade, "grade", "expected_loss_rate", "Expected Loss Rate by Grade", "Expected Loss Rate", "expected_loss_rate_by_grade_v1.png")
    simple_bar(el_by_grade, "grade", "mean_predicted_pd", "Mean Predicted PD by Grade", "Mean Predicted PD", "mean_pd_by_grade_v1.png")
else:
    print("grade not available. Skipping grade output.")

## 6. Expected Loss by Sub-Grade

In [ ]:
if "sub_grade" in loans.columns:
    el_by_sub_grade = sort_segment_summary(segment_expected_loss_summary(loans, "sub_grade"), "sub_grade")
    save_segment_output(el_by_sub_grade, "expected_loss_by_sub_grade_v1.csv")
    display(el_by_sub_grade)
    simple_bar(el_by_sub_grade, "sub_grade", "expected_loss_rate", "Expected Loss Rate by Sub-Grade", "Expected Loss Rate", "expected_loss_rate_by_sub_grade_v1.png")
    simple_bar(el_by_sub_grade, "sub_grade", "total_expected_loss", "Total Expected Loss by Sub-Grade", "Total Expected Loss", "expected_loss_total_by_sub_grade_v1.png")
else:
    print("sub_grade not available. Skipping sub-grade output.")

## 7. Expected Loss by Term

In [ ]:
if "term" in loans.columns:
    el_by_term = sort_segment_summary(segment_expected_loss_summary(loans, "term"), "term")
    save_segment_output(el_by_term, "expected_loss_by_term_v1.csv")
    display(el_by_term)
    simple_bar(el_by_term, "term", "total_expected_loss", "Total Expected Loss by Term", "Total Expected Loss", "expected_loss_total_by_term_v1.png")
    simple_bar(el_by_term, "term", "expected_loss_rate", "Expected Loss Rate by Term", "Expected Loss Rate", "expected_loss_rate_by_term_v1.png")
else:
    print("term not available. Skipping term output.")

## 8. Expected Loss by Purpose

In [ ]:
if "purpose" in loans.columns:
    el_by_purpose = sort_segment_summary(segment_expected_loss_summary(loans, "purpose"), "purpose")
    save_segment_output(el_by_purpose, "expected_loss_by_purpose_v1.csv")
    display(el_by_purpose)
    simple_bar(el_by_purpose, "purpose", "total_expected_loss", "Top 10 Purposes by Total Expected Loss", "Total Expected Loss", "expected_loss_total_by_top_purposes_v1.png", top_n=10)
    top_by_ead = el_by_purpose.sort_values("total_ead", ascending=False).head(10)
    simple_bar(top_by_ead, "purpose", "expected_loss_rate", "Expected Loss Rate for Top 10 Purposes by EAD", "Expected Loss Rate", "expected_loss_rate_top_purposes_by_ead_v1.png")
else:
    print("purpose not available. Skipping purpose output.")

## 9. Expected Loss by Issue Year

In [ ]:
if "issue_year" in loans.columns:
    el_by_issue_year = sort_segment_summary(segment_expected_loss_summary(loans, "issue_year"), "issue_year")
    save_segment_output(el_by_issue_year, "expected_loss_by_issue_year_v1.csv")
    display(el_by_issue_year)
    simple_bar(el_by_issue_year, "issue_year", "total_expected_loss", "Total Expected Loss by Issue Year", "Total Expected Loss", "expected_loss_total_by_issue_year_v1.png")
    simple_bar(el_by_issue_year, "issue_year", "expected_loss_rate", "Expected Loss Rate by Issue Year", "Expected Loss Rate", "expected_loss_rate_by_issue_year_v1.png")
    simple_bar(el_by_issue_year, "issue_year", "mean_predicted_pd", "Mean Predicted PD by Issue Year", "Mean Predicted PD", "mean_pd_by_issue_year_v1.png")
else:
    print("issue_year not available. Skipping issue-year output.")

## 10. PD Decile Analysis

In [ ]:
loans["pd_decile"] = pd.qcut(loans["predicted_pd"], q=10, labels=False, duplicates="drop") + 1

el_by_pd_decile = (
    loans.groupby("pd_decile", dropna=False)
    .agg(
        loan_count=("expected_loss_prelim", "size"),
        default_count=("default_flag", "sum"),
        realized_default_rate=("default_flag", "mean"),
        min_predicted_pd=("predicted_pd", "min"),
        max_predicted_pd=("predicted_pd", "max"),
        mean_predicted_pd=("predicted_pd", "mean"),
        total_ead=("ead_proxy", "sum"),
        total_expected_loss=("expected_loss_prelim", "sum"),
        mean_expected_loss=("expected_loss_prelim", "mean"),
    )
    .reset_index()
    .sort_values("pd_decile")
)
el_by_pd_decile["expected_loss_rate"] = el_by_pd_decile["total_expected_loss"] / el_by_pd_decile["total_ead"]
el_by_pd_decile["share_of_total_ead"] = el_by_pd_decile["total_ead"] / total_ead
el_by_pd_decile["share_of_total_expected_loss"] = el_by_pd_decile["total_expected_loss"] / total_expected_loss
el_by_pd_decile["cumulative_share_of_total_expected_loss"] = el_by_pd_decile["share_of_total_expected_loss"].cumsum()

save_segment_output(el_by_pd_decile, "expected_loss_by_pd_decile_v1.csv")
display(el_by_pd_decile)

simple_bar(el_by_pd_decile, "pd_decile", "expected_loss_rate", "Expected Loss Rate by PD Decile", "Expected Loss Rate", "expected_loss_rate_by_pd_decile_v1.png")
simple_bar(el_by_pd_decile, "pd_decile", "share_of_total_expected_loss", "Share of Total Expected Loss by PD Decile", "Share of Total Expected Loss", "expected_loss_share_by_pd_decile_v1.png")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(el_by_pd_decile["pd_decile"], el_by_pd_decile["realized_default_rate"], marker="o", label="Realized default rate")
plt.plot(el_by_pd_decile["pd_decile"], el_by_pd_decile["mean_predicted_pd"], marker="o", label="Mean predicted PD")
plt.title("Realized Default Rate vs Mean Predicted PD by Decile")
plt.xlabel("PD Decile")
plt.ylabel("Rate")
plt.xticks(range(1, 11))
plt.legend()
plt.tight_layout()
plt.savefig(figures_dir / "realized_vs_predicted_pd_by_decile_v1.png", dpi=150)
plt.show()

## 11. Expected Loss Concentration Analysis

In [ ]:
concentration = loans.sort_values("expected_loss_prelim", ascending=False).copy()
concentration["el_rank"] = np.arange(1, len(concentration) + 1)
concentration["cumulative_expected_loss"] = concentration["expected_loss_prelim"].cumsum()
concentration["cumulative_ead"] = concentration["ead_proxy"].cumsum()
concentration["cumulative_share_of_expected_loss"] = concentration["cumulative_expected_loss"] / total_expected_loss
concentration["cumulative_share_of_loans"] = concentration["el_rank"] / len(concentration)
concentration["cumulative_share_of_ead"] = concentration["cumulative_ead"] / total_ead

top_group_rows = []
for share_of_loans in [0.01, 0.05, 0.10, 0.20]:
    n_top = int(np.ceil(len(concentration) * share_of_loans))
    top_group = concentration.head(n_top)
    top_group_rows.append({
        "top_group": f"top_{int(share_of_loans * 100)}pct_loans_by_expected_loss",
        "loan_count": len(top_group),
        "share_of_loans": len(top_group) / len(concentration),
        "total_expected_loss": top_group["expected_loss_prelim"].sum(),
        "share_of_total_expected_loss": top_group["expected_loss_prelim"].sum() / total_expected_loss,
        "total_ead": top_group["ead_proxy"].sum(),
        "share_of_total_ead": top_group["ead_proxy"].sum() / total_ead,
        "mean_predicted_pd": top_group["predicted_pd"].mean(),
        "mean_ead": top_group["ead_proxy"].mean(),
        "mean_expected_loss": top_group["expected_loss_prelim"].mean(),
    })

expected_loss_top_concentration = pd.DataFrame(top_group_rows)
save_segment_output(expected_loss_top_concentration, "expected_loss_top_concentration_v1.csv")
display(expected_loss_top_concentration)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(concentration["cumulative_share_of_loans"], concentration["cumulative_share_of_expected_loss"])
plt.title("Cumulative Share of Expected Loss vs Cumulative Share of Loans")
plt.xlabel("Cumulative Share of Loans")
plt.ylabel("Cumulative Share of Expected Loss")
plt.tight_layout()
plt.savefig(figures_dir / "cumulative_el_vs_loans_v1.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(concentration["cumulative_share_of_ead"], concentration["cumulative_share_of_expected_loss"])
plt.title("Cumulative Share of Expected Loss vs Cumulative Share of EAD")
plt.xlabel("Cumulative Share of EAD")
plt.ylabel("Cumulative Share of Expected Loss")
plt.tight_layout()
plt.savefig(figures_dir / "cumulative_el_vs_ead_v1.png", dpi=150)
plt.show()

## 12. Driver Decomposition

In [ ]:
driver_correlations = pd.DataFrame({
    "pair": [
        "predicted_pd vs expected_loss_prelim",
        "ead_proxy vs expected_loss_prelim",
        "predicted_pd vs ead_proxy",
    ],
    "correlation": [
        loans["predicted_pd"].corr(loans["expected_loss_prelim"]),
        loans["ead_proxy"].corr(loans["expected_loss_prelim"]),
        loans["predicted_pd"].corr(loans["ead_proxy"]),
    ],
})
display(driver_correlations)

In [ ]:
def create_qcut_labels(series, q, prefix, low_label, high_label):
    """Create quantile labels while handling duplicate bin edges gracefully."""
    numeric_bins = pd.qcut(series, q=q, labels=False, duplicates="drop") + 1
    max_bin = int(numeric_bins.max())
    labels = {1: f"Q1_{low_label}", max_bin: f"Q{max_bin}_{high_label}"}
    return numeric_bins.map(lambda value: labels.get(int(value), f"Q{int(value)}"))

loans["predicted_pd_quartile"] = create_qcut_labels(loans["predicted_pd"], 4, "pd", "lowest_pd", "highest_pd")
loans["ead_proxy_quartile"] = create_qcut_labels(loans["ead_proxy"], 4, "ead", "lowest_ead", "highest_ead")

pd_ead_matrix = (
    loans.groupby(["predicted_pd_quartile", "ead_proxy_quartile"], dropna=False, observed=False)
    .agg(
        loan_count=("expected_loss_prelim", "size"),
        mean_predicted_pd=("predicted_pd", "mean"),
        mean_ead=("ead_proxy", "mean"),
        total_ead=("ead_proxy", "sum"),
        mean_expected_loss=("expected_loss_prelim", "mean"),
        total_expected_loss=("expected_loss_prelim", "sum"),
    )
    .reset_index()
)
pd_ead_matrix["share_of_total_expected_loss"] = pd_ead_matrix["total_expected_loss"] / total_expected_loss

save_segment_output(pd_ead_matrix, "expected_loss_pd_ead_matrix_v1.csv")
display(pd_ead_matrix)

In [ ]:
mean_el_heatmap = pd_ead_matrix.pivot(index="predicted_pd_quartile", columns="ead_proxy_quartile", values="mean_expected_loss")
total_el_heatmap = pd_ead_matrix.pivot(index="predicted_pd_quartile", columns="ead_proxy_quartile", values="total_expected_loss")

plt.figure(figsize=(8, 6))
if SEABORN_AVAILABLE:
    sns.heatmap(mean_el_heatmap, annot=True, fmt=".0f", cmap="Blues")
else:
    plt.imshow(mean_el_heatmap, aspect="auto")
    plt.colorbar(label="Mean Expected Loss")
    plt.xticks(range(len(mean_el_heatmap.columns)), mean_el_heatmap.columns, rotation=45, ha="right")
    plt.yticks(range(len(mean_el_heatmap.index)), mean_el_heatmap.index)
plt.title("Mean Expected Loss by PD Quartile and EAD Quartile")
plt.xlabel("EAD Quartile")
plt.ylabel("PD Quartile")
plt.tight_layout()
plt.savefig(figures_dir / "mean_el_pd_ead_heatmap_v1.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 6))
if SEABORN_AVAILABLE:
    sns.heatmap(total_el_heatmap, annot=True, fmt=".0f", cmap="Reds")
else:
    plt.imshow(total_el_heatmap, aspect="auto")
    plt.colorbar(label="Total Expected Loss")
    plt.xticks(range(len(total_el_heatmap.columns)), total_el_heatmap.columns, rotation=45, ha="right")
    plt.yticks(range(len(total_el_heatmap.index)), total_el_heatmap.index)
plt.title("Total Expected Loss by PD Quartile and EAD Quartile")
plt.xlabel("EAD Quartile")
plt.ylabel("PD Quartile")
plt.tight_layout()
plt.savefig(figures_dir / "total_el_pd_ead_heatmap_v1.png", dpi=150)
plt.show()

## 13. End-of-Notebook Output Checks

In [ ]:
output_check = pd.DataFrame([
    {"file": name, "path": str(info["path"]), "shape": info["shape"], "exists": info["path"].exists()}
    for name, info in saved_outputs.items()
])
display(output_check)

top_pd_decile_share = el_by_pd_decile.loc[el_by_pd_decile["pd_decile"] == el_by_pd_decile["pd_decile"].max(), "share_of_total_expected_loss"].iloc[0]
top_10_concentration_share = expected_loss_top_concentration.loc[
    expected_loss_top_concentration["top_group"] == "top_10pct_loans_by_expected_loss",
    "share_of_total_expected_loss",
].iloc[0]

print(f"Portfolio total EAD: {total_ead:,.2f}")
print(f"Portfolio total Expected Loss: {total_expected_loss:,.2f}")
print(f"Portfolio Expected Loss rate: {total_expected_loss / total_ead:.4f}")
print(f"Mean predicted PD: {loans['predicted_pd'].mean():.4f}")
print(f"Mean LGD: {loans['lgd_estimate'].mean():.4f}")
print(f"Top PD decile share of total Expected Loss: {top_pd_decile_share:.4f}")
print(f"Top 10% loan-level Expected Loss concentration share: {top_10_concentration_share:.4f}")